# Step 07: Pulsicity & Flow — Command Center

**Gaga Motion Analysis Pipeline** — behavioral metrics computed from `{segment}__lin_vel_rel_mag`  
as produced by NB06 (`notebooks/06_ultimate_kinematics.ipynb`) and stored in  
`derivatives/step_06_kinematics/{RUN_ID}__kinematics_master.parquet`.

**Metrics computed:** noise floor V · PSD diagnostic · SPARC · peak detection (PPM, IPI CV)  
**Architecture:** `docs/STEP_07_MISSION_PLAN.md` | **Backend:** `src/pulsicity.py`  
**Strict constraints:** no re-differentiation · no filtfilt on Measurement Signal · PCHIP gaps only for SPARC

---
**Run cells in order.** Sections 3–5 depend on Sections 1–2 being executed first.

In [ ]:
# ── Cell 1: Setup & Imports ───────────────────────────────────────────────
%matplotlib inline
import os
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

# Logging
logging.basicConfig(
    level=logging.WARNING,
    format='%(levelname)s | %(name)s | %(message)s',
)
logger = logging.getLogger('nb07')
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Project root discovery ────────────────────────────────────────────────
# Works whether CWD is project root, notebooks/, or any subdirectory.
_search = Path().resolve()
for _ in range(6):
    if (_search / 'config' / 'config_v1.yaml').exists():
        PROJECT_ROOT = _search
        break
    _search = _search.parent
else:
    raise FileNotFoundError(
        'Cannot locate project root (config/config_v1.yaml not found in any parent directory).'
    )

SRC_DIR    = PROJECT_ROOT / 'src'
STEP06_DIR = PROJECT_ROOT / 'derivatives' / 'step_06_kinematics'
STEP07_DIR = PROJECT_ROOT / 'derivatives' / 'step_07_behavioral'
STEP07_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Import pulsicity backend ──────────────────────────────────────────────
from pulsicity import (
    check_enforce_cleaning_provenance,
    get_inherited_config,
    compute_sg_effective_cutoff,
    compute_noise_floor,
    compute_psd_diagnostic,
    compute_sparc,
    detect_velocity_peaks,
    aggregate_pulsicity_metrics,
)

# ── Load pipeline config ──────────────────────────────────────────────────
from pipeline_config import CONFIG as _RAW_CFG
CFG = dict(_RAW_CFG)   # mutable notebook copy

# ── Discover Step 06 parquet files ────────────────────────────────────────
parquet_files = sorted(STEP06_DIR.glob('*__kinematics_master.parquet'))
run_ids = [f.stem.replace('__kinematics_master', '') for f in parquet_files]

# ── Global session state (populated by the Diagnostics cell) ─────────────
_diag_results  = {}   # seg -> {'nf':..., 'psd':..., 'sp':..., 'df':...}
_filter_state  = {}   # seg -> {'apply': bool, 'cutoff_hz': float}
_filter_widgets = {}  # seg -> {'checkbox': widget, 'slider': widget}

# ── Summary ───────────────────────────────────────────────────────────────
print(f'Project root : {PROJECT_ROOT}')
print(f'Step 06 dir  : {STEP06_DIR}')
print(f'Step 07 dir  : {STEP07_DIR}')
print(f'Recordings   : {len(run_ids)} parquet files')
for _rid in run_ids:
    print(f'  {_rid}')

In [ ]:
import sys
import subprocess

# 1. Force install using the exact Python executable running this notebook
print(f"Installing into: {sys.executable}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "fastparquet", "--upgrade", "ipywidgets", "jupyterlab_widgets"])

# 2. Test the import directly to see if it throws a specific error
try:
    import pyarrow
    print("✅ PyArrow imported successfully!")
except Exception as e:
    print(f"❌ PyArrow failed to import. The real error is: {e}")

## Section 1 — Config Audit Block

Validates that the Step 06 parameters inherited by Step 07 match `config/config_v1.yaml`.  
Sets the global `F_EFF` (SavGol effective cutoff — SPARC cap and PSD reference frequency).

In [ ]:
# ── Cell 2: Section 1 — Config Audit Block ───────────────────────────────
inherited = get_inherited_config(CFG)
F_EFF     = inherited['sg_eff_cutoff_hz']   # global: used by all downstream cells

_fs   = inherited['fs_target']
_wsec = inherited['sg_window_sec']
_wlen = inherited['sg_window_len']
_poly = inherited['sg_polyorder']
_feff = inherited['sg_eff_cutoff_hz']
_rss  = inherited['ref_search_sec']
_rws  = inherited['ref_window_sec']
_mrs  = inherited['min_run_seconds']
_ec   = inherited['enforce_cleaning']

_SEP = '\u2501' * 56   # ━━━━━━━━━
print(_SEP)
print('STEP 07  \u2014  CONFIG INHERITANCE AUDIT')
print(_SEP)
print(f'  fs_target       : {_fs:.1f} Hz        [config/config_v1.yaml]')
print(f'  sg_window_sec   : {_wsec:.3f} s  \u2192  {_wlen} frames')
print(f'  sg_polyorder    : {_poly}')
print(f'  sg_eff_cutoff   : {_feff:.4f} Hz  (99%-amplitude threshold \u2014 SPARC cap)')
print(f'  ref_search_sec  : {_rss:.1f} s')
print(f'  ref_window_sec  : {_rws:.1f} s')
print(f'  min_run_seconds : {_mrs:.1f} s')
print(f'  enforce_cleaning: {_ec}')
print(_SEP)
print(f'  Target segments : LeftHand, RightHand, LeftFoot, RightFoot, Head')
print(f'  Input parquets  : derivatives/step_06_kinematics/  [{len(run_ids)} recordings]')
print(f'  Output dir      : derivatives/step_07_behavioral/')
print(_SEP)

if _ec:
    print()
    print('\u26a0  PROVENANCE WARNING: step_06.enforce_cleaning = True')
    print('   lin_vel_rel_mag may be STALE for surgically repaired segments.')
    print('   See docs/STEP_0_AUDIT_REPORT.md  Section 3 Finding F2.')
    print()

## Section 2 — Interactive Parameter Overrides

These parameters are **Step 07–specific** (not in `config_v1.yaml`).  
Defaults are from `STEP_07_MISSION_PLAN.md §2.3`.  
Changes here propagate to Section 3/4 (diagnostics) and Section 5 (visual audit) automatically.

In [ ]:
# ── Cell 3: Section 2 — Interactive Parameter Overrides ──────────────────
DEFAULT_SEGMENTS = ['LeftHand', 'RightHand', 'LeftFoot', 'RightFoot', 'Head']

# Discover all segments present in the first available parquet
if parquet_files:
    _peek_cols = pd.read_parquet(parquet_files[0]).columns
    ALL_SEGMENTS = sorted(
        c.split('__')[0]
        for c in _peek_cols
        if c.endswith('__lin_vel_rel_mag')
    )
    del _peek_cols
else:
    ALL_SEGMENTS = DEFAULT_SEGMENTS

_W_SLIDER  = widgets.Layout(width='500px')
_W_LABEL   = {'description_width': '145px'}

w_recording = widgets.Dropdown(
    options=run_ids,
    value=run_ids[0] if run_ids else None,
    description='Recording:',
    layout=widgets.Layout(width='700px'),
    style={'description_width': '110px'},
)

w_segments = widgets.SelectMultiple(
    options=ALL_SEGMENTS,
    value=[s for s in DEFAULT_SEGMENTS if s in ALL_SEGMENTS],
    description='Segments:',
    layout=widgets.Layout(width='290px', height='160px'),
    style={'description_width': '90px'},
)

w_noise_guard = widgets.FloatSlider(
    value=1.0, min=0.0, max=50.0, step=0.5,
    description='V floor (mm/s):',
    continuous_update=False,
    layout=_W_SLIDER, style=_W_LABEL, readout_format='.1f',
)

w_prominence = widgets.FloatSlider(
    value=0.5, min=0.1, max=2.0, step=0.05,
    description='Prominence \u00d7\u03c3:',
    continuous_update=False,
    layout=_W_SLIDER, style=_W_LABEL, readout_format='.2f',
)

w_min_dist_ms = widgets.IntSlider(
    value=100, min=50, max=500, step=10,
    description='Min dist (ms):',
    continuous_update=False,
    layout=_W_SLIDER, style=_W_LABEL,
)

w_static_guard = widgets.FloatSlider(
    value=50.0, min=0.0, max=200.0, step=5.0,
    description='Static guard (mm/s):',
    continuous_update=False,
    layout=_W_SLIDER, style=_W_LABEL, readout_format='.0f',
)

w_height_gate = widgets.Checkbox(
    value=True,
    description='Apply noise floor height gate  (height=V in find_peaks)',
    indent=False,
    layout=widgets.Layout(width='500px'),
)

print('\u2501' * 56)
print('SECTION 2  \u2014  STEP 07 PARAMETER OVERRIDES')
print('\u2501' * 56)
print('Defaults from STEP_07_MISSION_PLAN.md \u00a72.3.')
print()
display(widgets.VBox([
    w_recording,
    widgets.HBox([
        w_segments,
        widgets.VBox([
            w_noise_guard,
            w_prominence,
            w_min_dist_ms,
            w_static_guard,
            w_height_gate,
        ]),
    ]),
]))

## Section 3 — Diagnostics Panel  +  Section 4 — Human-Triggered Filter Gate

**Section 3:** Loads the selected recording, runs `compute_noise_floor`, `compute_psd_diagnostic`,  
and `compute_sparc` for each selected segment. Plots the Welch PSD with the SavGol effective  
cutoff line (`f_eff ≈ 2.4 Hz`) and colour-coded noise-ratio banner.

**Section 4:** If any segment has `psd_filter_recommended = True`, presents the **human-triggered  
Butterworth filter gate**. The researcher must explicitly approve the secondary filter before  
peak detection runs. The Measurement Signal `v_m` is never modified by this filter.

In [ ]:
# ── Cell 4: Section 3 & 4 — Diagnostics Panel & Filter Gate ─────────────
_DEFAULT_SEC_CUTOFF = 1.0
_BANNER_ICON = {'green': '\u2705', 'yellow': '\u26a0\ufe0f', 'red': '\U0001f534', 'unknown': '\u2753'}

run_diag_btn  = widgets.Button(
    description='\u25b6  Run Diagnostics',
    button_style='primary',
    tooltip='Load parquet and run noise floor, PSD, SPARC for all selected segments',
    layout=widgets.Layout(width='220px'),
)
diag_output   = widgets.Output()
filter_output = widgets.Output()


def _on_run_diagnostics(_btn):
    global _diag_results, _filter_state, _filter_widgets

    run_id   = w_recording.value
    segments = list(w_segments.value)
    _fs      = inherited['fs_target']

    if not run_id:
        with diag_output:
            print('No recording selected.')
        return

    _ppath = STEP06_DIR / f'{run_id}__kinematics_master.parquet'
    if not _ppath.exists():
        with diag_output:
            print(f'ERROR: {_ppath} not found.')
        return

    df = pd.read_parquet(_ppath)
    _diag_results.clear()
    _filter_state.clear()
    _filter_widgets.clear()

    # ── Section 3: PSD plots ──────────────────────────────────────────────
    with diag_output:
        clear_output(wait=True)
        _dur_s = len(df) / _fs
        print(f'Recording : {run_id}')
        print(f'Frames    : {len(df)}  ({_dur_s:.1f} s @ {_fs:.0f} Hz)')
        print(f'Segments  : {segments}')
        print()

        _valid = [s for s in segments if f'{s}__lin_vel_rel_mag' in df.columns]
        _miss  = [s for s in segments if s not in _valid]
        if _miss:
            print(f'\u26a0  Missing segments (skipped): {_miss}')
        if not _valid:
            print('No valid segments found in parquet.')
            return

        _ncols = len(_valid)
        _fig, _axes = plt.subplots(1, _ncols, figsize=(5 * _ncols, 4), squeeze=False)
        _fig.suptitle(f'PSD Diagnostic  \u2014  {run_id}', fontsize=11, fontweight='bold')

        for _i, seg in enumerate(_valid):
            _ax = _axes[0][_i]

            # noise floor
            try:
                nf = compute_noise_floor(
                    df, seg, CFG,
                    static_baseline_guard_mms=w_static_guard.value,
                    noise_floor_guard_mms=w_noise_guard.value,
                )
            except Exception as _exc:
                _ax.text(0.5, 0.5, f'{seg}\nnoise floor error:\n{_exc}',
                         ha='center', va='center', transform=_ax.transAxes,
                         color='red', fontsize=8)
                print(f'{seg}: noise floor ERROR  {_exc}')
                continue

            # PSD
            try:
                psd = compute_psd_diagnostic(df, seg, CFG, f_eff=F_EFF)
            except Exception as _exc:
                _ax.text(0.5, 0.5, f'{seg}\nPSD error:\n{_exc}',
                         ha='center', va='center', transform=_ax.transAxes,
                         color='red', fontsize=8)
                print(f'{seg}: PSD ERROR  {_exc}')
                continue

            # SPARC
            try:
                sp = compute_sparc(df, seg, CFG, f_eff=F_EFF)
            except Exception as _exc:
                sp = {'sparc': float('nan'), 'sparc_failed': True,
                      'sparc_failure_reason': str(_exc)}

            _diag_results[seg] = {'nf': nf, 'psd': psd, 'sp': sp, 'df': df}
            _filter_state[seg] = {'apply': False, 'cutoff_hz': _DEFAULT_SEC_CUTOFF}

            # ── PSD plot ──────────────────────────────────────────────────
            _freqs, _pw = psd['freqs'], psd['psd']
            if len(_freqs) > 0:
                _ax.semilogy(_freqs, _pw, color='steelblue', lw=1.5, zorder=4,
                             label='Welch PSD')
                _ax.axvline(F_EFF, color='tomato', ls='--', lw=1.8, zorder=5,
                            label=f'f_eff = {F_EFF:.2f} Hz')
                _sig_m  = (_freqs >= 0.5) & (_freqs <  F_EFF)
                _noi_m  = (_freqs >= F_EFF) & (_freqs <= 10.0)
                if _sig_m.any():
                    _ax.fill_between(_freqs[_sig_m],  _pw[_sig_m],  alpha=0.18, color='steelblue')
                if _noi_m.any():
                    _ax.fill_between(_freqs[_noi_m],  _pw[_noi_m],  alpha=0.18, color='tomato')

            _banner = psd['psd_banner_level']
            _icon   = _BANNER_ICON.get(_banner, '?')
            _nr     = psd['noise_ratio']
            _nr_str = f'{_nr:.3f}' if not np.isnan(_nr) else 'N/A'
            _ax.set_title(f'{seg}\n{_icon}  noise_ratio = {_nr_str}', fontsize=9)
            _ax.set_xlabel('Frequency (Hz)', fontsize=8)
            _ax.set_ylabel('PSD', fontsize=8)
            _ax.set_xlim(0, 15)
            if len(_freqs) > 0:
                _ax.legend(fontsize=7)
            _ax.tick_params(labelsize=7)

            # ── Console summary ────────────────────────────────────────────
            _v_val  = nf['V']
            _nfsrc  = nf['noise_floor_source']
            _lc     = nf['noise_floor_low_confidence']
            _sp_val = sp['sparc']
            _sp_fail= sp.get('sparc_failed', False)
            _sp_str = f'{_sp_val:.4f}' if not _sp_fail else f'FAILED ({sp.get("sparc_failure_reason", "?")})'  # noqa
            _lc_str = '  \u26a0 LOW_CONFIDENCE' if _lc else ''
            print(f'{seg}:')
            print(f'  V            = {_v_val:.2f} mm/s  |  {_nfsrc}{_lc_str}')
            print(f'  noise_ratio  = {_nr_str}  [{_banner}]')
            print(f'  SPARC        = {_sp_str}')
            print()

        plt.tight_layout()
        plt.show()

    # ── Section 4: Filter gate ────────────────────────────────────────────
    with filter_output:
        clear_output(wait=True)
        _rec_segs = [
            s for s, r in _diag_results.items()
            if r['psd'].get('psd_filter_recommended', False)
        ]
        if not _rec_segs:
            print('\u2705  No secondary filter recommended for any segment.')
            return

        print('\u2501' * 60)
        print('SECTION 4  \u2014  HUMAN-TRIGGERED FILTER GATE')
        print('\u2501' * 60)
        print('Secondary Butterworth filter recommended for:')
        for _s in _rec_segs:
            _nr_s = _diag_results[_s]['psd']['noise_ratio']
            print(f'  {_s}  (noise_ratio = {_nr_s:.3f})')
        print()
        print('Choose which segments to filter and confirm the cutoff.')
        print('Filter applies to the Search Signal only  \u2014  v_m is NEVER modified.')
        print()

        _rows = []
        for _seg in sorted(_diag_results.keys()):
            _cb = widgets.Checkbox(
                value=(_seg in _rec_segs),
                description=_seg,
                indent=False,
                layout=widgets.Layout(width='160px'),
            )
            _sl = widgets.FloatSlider(
                value=_DEFAULT_SEC_CUTOFF,
                min=0.3, max=round(float(F_EFF), 1), step=0.1,
                description='Cutoff (Hz):',
                continuous_update=False,
                layout=widgets.Layout(width='380px'),
                style={'description_width': '100px'},
                readout_format='.1f',
                disabled=not _cb.value,
            )

            def _toggle_sl(change, _s=_sl):
                _s.disabled = not change['new']

            _cb.observe(_toggle_sl, names='value')
            _filter_widgets[_seg] = {'checkbox': _cb, 'slider': _sl}
            _rows.append(widgets.HBox([_cb, _sl]))

        _apply_btn = widgets.Button(
            description='\u2714  Confirm Choices',
            button_style='success',
            layout=widgets.Layout(width='180px'),
        )
        _skip_btn = widgets.Button(
            description='Skip All',
            button_style='warning',
            layout=widgets.Layout(width='120px'),
        )
        _filter_note = widgets.Output()

        def _on_apply(_b):
            with _filter_note:
                clear_output(wait=True)
                for _s, _wd in _filter_widgets.items():
                    _filter_state[_s]['apply']     = _wd['checkbox'].value
                    _filter_state[_s]['cutoff_hz'] = _wd['slider'].value
                _parts = []
                for _s in sorted(_filter_widgets):
                    _app = _filter_state[_s]['apply']
                    _chz = _filter_state[_s]['cutoff_hz']
                    _parts.append(f'{_s}@{_chz:.1f}Hz' if _app else f'{_s}:skip')
                print(f'\u2705  Filter choices saved: {", ".join(_parts)}')

        def _on_skip(_b):
            with _filter_note:
                clear_output(wait=True)
                for _s in _filter_state:
                    _filter_state[_s]['apply'] = False
                for _s, _wd in _filter_widgets.items():
                    _wd['checkbox'].value = False
                print('\u2705  All secondary filters skipped.')

        _apply_btn.on_click(_on_apply)
        _skip_btn.on_click(_on_skip)
        display(widgets.VBox(
            _rows + [widgets.HBox([_apply_btn, _skip_btn]), _filter_note]
        ))


run_diag_btn.on_click(_on_run_diagnostics)

print('\u2501' * 56)
print('SECTION 3  \u2014  DIAGNOSTICS PANEL')
print('\u2501' * 56)
print('Select a recording and segments above, then click Run Diagnostics.')
display(widgets.VBox([run_diag_btn, diag_output, filter_output]))

## Section 5 — Windowed Visual Audit

Interactive signal inspector. Pan through the full recording to verify peak detection quality.

| Colour | Signal |
|--------|--------|
| **Blue** | Measurement Signal `v_m(t)` — ground-truth magnitude (NaN on artifact frames → natural gaps) |
| **Orange** | Search Signal `v_s(t)` — only shown if secondary Butterworth was applied |
| **Red ▼** | Detected peaks (inverted triangles) — `v_m` values at accepted peak frames |
| **Dashed orange** | Noise floor `V` (adaptive threshold) |
| **Grey shading** | Artifact frames (`is_artifact == True`) |

> **Run Section 3 first** (click *Run Diagnostics*) to populate segment data before using this panel.

In [ ]:
# ── Cell 5: Section 5 — Windowed Visual Audit ────────────────────────────

w_audit_seg = widgets.Dropdown(
    description='Segment:',
    options=[],
    layout=widgets.Layout(width='230px'),
    style={'description_width': '80px'},
)

w_start_s = widgets.FloatSlider(
    value=0.0, min=0.0, max=260.0, step=0.5,
    description='Start (s):',
    continuous_update=False,
    layout=widgets.Layout(width='750px'),
    style={'description_width': '90px'},
    readout_format='.1f',
)

w_win_dur = widgets.FloatSlider(
    value=10.0, min=2.0, max=60.0, step=0.5,
    description='Window (s):',
    continuous_update=False,
    layout=widgets.Layout(width='750px'),
    style={'description_width': '90px'},
    readout_format='.1f',
)

w_show_vs    = widgets.Checkbox(
    value=True, description='Show v_s (Search Signal)',
    indent=False, layout=widgets.Layout(width='250px'),
)
w_show_peaks = widgets.Checkbox(
    value=True, description='Show detected peaks',
    indent=False, layout=widgets.Layout(width='220px'),
)
w_show_gaps  = widgets.Checkbox(
    value=True, description='Mark NaN gap boundaries',
    indent=False, layout=widgets.Layout(width='240px'),
)

audit_out = widgets.Output()


def _refresh_seg_opts():
    _opts = list(_diag_results.keys())
    w_audit_seg.options = _opts
    if _opts and w_audit_seg.value not in _opts:
        w_audit_seg.value = _opts[0]


def _plot_audit(_change=None):
    _seg = w_audit_seg.value
    if not _seg or _seg not in _diag_results:
        with audit_out:
            clear_output(wait=True)
            print('\u2139  Run diagnostics first (Section 3 button above).')
        return

    _fs  = inherited['fs_target']
    _df  = _diag_results[_seg]['df']
    _nf  = _diag_results[_seg]['nf']
    _sp  = _diag_results[_seg]['sp']
    _psd = _diag_results[_seg]['psd']

    _V       = _nf['V']
    _n_fr    = len(_df)
    _total_s = _n_fr / _fs

    # Clamp slider ranges to actual recording length
    w_start_s.max = max(0.0, round(_total_s - w_win_dur.value, 1))

    _st_fr = int(w_start_s.value * _fs)
    _en_fr = min(_n_fr, int((w_start_s.value + w_win_dur.value) * _fs))
    if _en_fr <= _st_fr:
        _en_fr = min(_n_fr, _st_fr + int(_fs))

    _df_win = _df.iloc[_st_fr:_en_fr]

    # Secondary filter state for this segment
    _sec_app = _filter_state.get(_seg, {}).get('apply', False)
    _sec_hz  = _filter_state.get(_seg, {}).get('cutoff_hz', None) if _sec_app else None

    # ── Run peak detection on the window ──────────────────────────────────
    _min_dist_fr = max(1, int(w_min_dist_ms.value / 1000.0 * _fs))
    try:
        _pk = detect_velocity_peaks(
            _df_win, _seg, CFG,
            V=_V,
            prominence_multiplier=w_prominence.value,
            min_distance_frames=_min_dist_fr,
            height_gate=w_height_gate.value,
            secondary_filter_cutoff_hz=_sec_hz,
        )
    except Exception as _exc:
        with audit_out:
            clear_output(wait=True)
            print(f'ERROR in detect_velocity_peaks({_seg}): {_exc}')
        return

    _v_m      = _pk['v_m']
    _v_s      = _pk['v_s']
    _artifact = _pk['artifact']
    _pk_idx   = _pk['peak_indices']    # frame indices within df_win

    # Absolute time axis for this window
    _t_ax = np.arange(_st_fr, _en_fr) / _fs

    with audit_out:
        clear_output(wait=True)
        _fig, _ax = plt.subplots(figsize=(16, 5))
        _ax.set_facecolor('#f8f8f8')

        # ── Artifact shading (grey) ────────────────────────────────────────
        _pad_art   = np.concatenate([[False], _artifact, [False]])
        _art_starts = np.where(~_pad_art[:-1] &  _pad_art[1:])[0]
        _art_ends   = np.where( _pad_art[:-1] & ~_pad_art[1:])[0]
        for _as, _ae in zip(_art_starts, _art_ends):
            _t0 = _t_ax[min(_as,     len(_t_ax) - 1)]
            _t1 = _t_ax[min(_ae - 1, len(_t_ax) - 1)]
            _ax.axvspan(_t0, _t1, color='lightgray', alpha=0.65, zorder=1)

        # ── Noise floor V (dashed orange) ─────────────────────────────────
        _nfsrc_short = _nf['noise_floor_source'][:30]
        _ax.axhline(_V, color='darkorange', ls='--', lw=1.6, zorder=2,
                    label=f'V = {_V:.1f} mm/s  [{_nfsrc_short}]')

        # ── Search Signal v_s (orange) — only when secondary filter active ─
        if w_show_vs.value and _sec_app and _sec_hz is not None:
            _ax.plot(_t_ax, _v_s, color='darkorange', lw=1.0, alpha=0.60, zorder=3,
                     label=f'v_s  (Butterworth {_sec_hz:.1f} Hz)')

        # ── Measurement Signal v_m (blue) ──────────────────────────────────
        # NaN values (artifact spans) produce natural breaks in the plot line.
        _ax.plot(_t_ax, _v_m, color='steelblue', lw=1.3, zorder=4,
                 label='v_m  (Measurement Signal)')

        # ── NaN gap boundary markers (dotted crimson verticals) ────────────
        if w_show_gaps.value and np.isnan(_v_m).any():
            _pad_nan   = np.concatenate([[False], np.isnan(_v_m), [False]])
            _nan_starts = np.where(~_pad_nan[:-1] &  _pad_nan[1:])[0]
            _nan_ends   = np.where( _pad_nan[:-1] & ~_pad_nan[1:])[0]
            for _ns in _nan_starts:
                if _ns < len(_t_ax):
                    _ax.axvline(_t_ax[_ns], color='crimson', ls=':', lw=0.9,
                                alpha=0.45, zorder=3)
            for _ne in _nan_ends:
                _idx = min(_ne, len(_t_ax) - 1)
                _ax.axvline(_t_ax[_idx], color='crimson', ls=':', lw=0.9,
                            alpha=0.45, zorder=3)

        # ── Detected peaks (red inverted triangles) ────────────────────────
        if w_show_peaks.value and len(_pk_idx) > 0:
            _pk_t = _t_ax[_pk_idx]
            _pk_v = _pk['peak_velocities_mms']
            _ax.scatter(_pk_t, _pk_v, marker='v', color='crimson', s=90, zorder=6,
                        label=f'Peaks  N\u202f=\u202f{len(_pk_idx)}')

        # ── Axis labels & title ────────────────────────────────────────────
        _ax.set_xlabel('Time (s)', fontsize=11)
        _ax.set_ylabel('Speed  (mm/s)', fontsize=11)
        _ax.set_xlim(_t_ax[0], _t_ax[-1])

        _sp_val    = _sp['sparc']
        _sp_failed = _sp.get('sparc_failed', False)
        _sparc_str = f'{_sp_val:.3f}' if not _sp_failed else 'N/A'
        _nr_val    = _psd['noise_ratio']
        _nr_str    = f'{_nr_val:.3f}' if not np.isnan(_nr_val) else 'N/A'
        _bann      = _psd['psd_banner_level']
        _filt_str  = f'  |  Butterworth {_sec_hz:.1f} Hz' if _sec_app and _sec_hz else ''
        _t_start   = w_start_s.value
        _t_end     = w_start_s.value + w_win_dur.value
        _ax.set_title(
            f'{_seg}  \u2014  {w_recording.value}  '
            f'[{_t_start:.1f} \u2013 {_t_end:.1f} s]\n'
            f'SPARC = {_sparc_str}  |  noise_ratio = {_nr_str} [{_bann}]{_filt_str}',
            fontsize=10,
        )

        # ── Legend + artifact patch ────────────────────────────────────────
        _art_patch = mpatches.Patch(color='lightgray', alpha=0.7, label='Artifact')
        _h, _l = _ax.get_legend_handles_labels()
        _ax.legend(_h + [_art_patch], _l + ['Artifact'],
                   fontsize=9, loc='upper right')

        # ── Stats annotation (bottom-left) ─────────────────────────────────
        _n_art    = int(np.sum(_artifact))
        _art_pct  = 100.0 * _n_art / len(_artifact) if len(_artifact) > 0 else 0.0
        _prom_thr = _pk['prominence_threshold_mms']
        _bridged  = _pk['n_bridged_frames_search']
        _sigma_v  = _pk['sigma_v_mms']
        _stats = (
            f'n_peaks = {len(_pk_idx)}  |  '
            f'\u03c3_v = {_sigma_v:.1f} mm/s  |  '
            f'prom_thr = {_prom_thr:.1f} mm/s  |  '
            f'bridged = {_bridged} fr  |  '
            f'artifact = {_art_pct:.1f}%'
        )
        _ax.text(0.005, 0.02, _stats, transform=_ax.transAxes,
                 fontsize=8, va='bottom', color='#333333',
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.75))

        plt.tight_layout()
        plt.show()


# Wire up widget observers
for _w in [w_audit_seg, w_start_s, w_win_dur,
           w_show_vs, w_show_peaks, w_show_gaps]:
    _w.observe(_plot_audit, names='value')

# Re-draw when parameter sliders change
for _w in [w_prominence, w_min_dist_ms, w_height_gate]:
    _w.observe(_plot_audit, names='value')

refresh_btn = widgets.Button(
    description='\u21ba  Refresh',
    button_style='',
    layout=widgets.Layout(width='120px'),
)
refresh_btn.on_click(_plot_audit)

# Refresh segment dropdown when new diagnostics are run
def _sync_seg_opts(_b=None):
    _refresh_seg_opts()
    _plot_audit()

run_diag_btn.on_click(_sync_seg_opts)   # piggyback on the diagnostics button

print('\u2501' * 56)
print('SECTION 5  \u2014  WINDOWED VISUAL AUDIT')
print('\u2501' * 56)
display(widgets.VBox([
    widgets.HBox([w_audit_seg, w_show_vs, w_show_peaks, w_show_gaps, refresh_btn]),
    w_start_s,
    w_win_dur,
    audit_out,
]))
_refresh_seg_opts()
_plot_audit()

## Section 6 — Batch Processing & Parquet Export

Runs the full Step 07 pipeline over every recording in `derivatives/step_06_kinematics/`.  
One output Parquet per recording → `derivatives/step_07_behavioral/{RUN_ID}__pulsicity_metrics.parquet`.

**Parameter inheritance:** prominence, min-distance, height-gate, and noise-guard come from  
the Section 2 widgets. `BATCH_SECONDARY_FILTER_HZ` is the only batch-specific override.

In [ ]:
# ── Cell 6: Section 6 — Batch Processing & Parquet Export ────────────────

# ── Batch secondary-filter override ───────────────────────────────────────
# 8.0 Hz preserves genuine rapid Gaga movement (~3–6 Hz peak motor rate)
# while attenuating electronic tracking spikes that alias above human motor
# bandwidth. Researcher may adjust upward to 15.0 Hz for very fast movement;
# do not go below ~3 Hz or rapid gestures will be suppressed.
BATCH_SECONDARY_FILTER_HZ = 8.0   # Hz  ← tune between 3.0–15.0 for Gaga

# ── Batch constants ────────────────────────────────────────────────────────
_BATCH_SEGS    = ['LeftHand', 'RightHand', 'LeftFoot', 'RightFoot', 'Head']
_fs_b          = inherited['fs_target']
_min_dist_fr_b = max(1, int(w_min_dist_ms.value / 1000.0 * _fs_b))
_enforce_was   = check_enforce_cleaning_provenance(CFG)

def _fmt(val, spec='.1f'):
    try:
        return 'NaN' if (isinstance(val, float) and np.isnan(val)) else format(val, spec)
    except (TypeError, ValueError):
        return str(val)

_SEP = '━' * 68
print(_SEP)
print('SECTION 6  —  BATCH PROCESSING & PARQUET EXPORT')
print(_SEP)
print(f'  Secondary filter  : {BATCH_SECONDARY_FILTER_HZ:.1f} Hz (Butterworth, Search Signal only)')
print(f'  Prominence ×σ     : {w_prominence.value:.2f}  (Section 2 widget)')
print(f'  Min distance      : {_min_dist_fr_b} frames  ({w_min_dist_ms.value} ms)')
print(f'  Height gate       : {w_height_gate.value}')
print(f'  Noise floor guard : {w_noise_guard.value:.1f} mm/s')
print(f'  Static guard      : {w_static_guard.value:.1f} mm/s')
print(f'  enforce_cleaning  : {_enforce_was}')
print(f'  Recordings        : {len(parquet_files)}')
print(f'  Segments          : {_BATCH_SEGS}')
print(_SEP)
print()

_all_rows = []

for _ppath in parquet_files:
    _run_id = _ppath.stem.replace('__kinematics_master', '')
    print(f'▶  {_run_id}')

    try:
        _df_b = pd.read_parquet(_ppath)
    except Exception as _exc:
        print(f'   ERROR reading parquet: {_exc}')
        continue

    _run_rows = []

    for _seg in _BATCH_SEGS:
        _vcol = f'{_seg}__lin_vel_rel_mag'
        if _vcol not in _df_b.columns:
            print(f'   {_seg:14s}  SKIP (column not found)')
            continue

        try:
            # Step 1 — Noise floor (session-adaptive threshold V)
            _nf_b = compute_noise_floor(
                _df_b, _seg, CFG,
                static_baseline_guard_mms=w_static_guard.value,
                noise_floor_guard_mms=w_noise_guard.value,
            )

            # Step 2a — PSD diagnostic (informational; batch never blocks on this)
            _psd_b = compute_psd_diagnostic(_df_b, _seg, CFG, f_eff=F_EFF)

            # Step 2b — SPARC smoothness metric (PCHIP gap bridging)
            _sp_b = compute_sparc(_df_b, _seg, CFG, f_eff=F_EFF)

            # Step 3 — Peak detection (Search Signal filtered at BATCH_SECONDARY_FILTER_HZ)
            _pk_b = detect_velocity_peaks(
                _df_b, _seg, CFG,
                V=_nf_b['V'],
                prominence_multiplier=w_prominence.value,
                min_distance_frames=_min_dist_fr_b,
                height_gate=w_height_gate.value,
                secondary_filter_cutoff_hz=BATCH_SECONDARY_FILTER_HZ,
            )

            # Step 4 — Aggregate into §3.3 output schema row
            _row = aggregate_pulsicity_metrics(
                _df_b, _seg, CFG,
                peaks_result=_pk_b,
                V=_nf_b['V'],
                sparc_result=_sp_b,
                psd_result=_psd_b,
                noise_floor_result=_nf_b,
                secondary_filter_applied=True,
                secondary_filter_cutoff_hz=BATCH_SECONDARY_FILTER_HZ,
                enforce_cleaning_was_active=_enforce_was,
                run_id=_run_id,
            )

            _run_rows.append(_row)
            _all_rows.append(_row)

            _vmf = '✓' if _row.get('valid_movement_flag') else '✗'
            print(
                f'   {_seg:14s}  '
                f'PPM={_fmt(_row["ppm"], ".1f"):>6}  '
                f'IPI_CV={_fmt(_row.get("ipi_cv", float("nan")), ".3f"):>7}  '
                f'SPARC={_fmt(_row["sparc"], ".3f"):>8}  '
                f'T_a={_fmt(_row.get("active_time_s", float("nan")), ".1f"):>6}s  '
                f'N_p={_row.get("n_peaks", "?"):>4}  valid={_vmf}'
            )

        except Exception as _exc:
            import traceback as _tb
            print(f'   {_seg:14s}  ERROR — {_exc}')
            logger.debug(_tb.format_exc())
            continue

    # ── Save per-recording Parquet ─────────────────────────────────────────
    if _run_rows:
        _out_path = STEP07_DIR / f'{_run_id}__pulsicity_metrics.parquet'
        try:
            pd.DataFrame(_run_rows).to_parquet(_out_path, index=False)
            print(f'   → {_out_path.name}  ({len(_run_rows)} rows)')
        except Exception as _exc:
            print(f'   ERROR saving parquet: {_exc}')
    else:
        print('   ⚠  No valid segments — parquet not written.')
    print()

# ── Batch summary DataFrame ────────────────────────────────────────────────
if _all_rows:
    batch_df = pd.DataFrame(_all_rows)
    print(_SEP)
    print(f'BATCH COMPLETE  —  {len(batch_df)} segment×recording rows  |  '
          f'{len(parquet_files)} recordings  |  Parquets → {STEP07_DIR.name}/')
    print(_SEP)
    _display_cols = [
        'run_id', 'segment', 'ppm', 'ipi_cv', 'sparc',
        'active_time_s', 'n_peaks', 'valid_movement_flag',
        'noise_floor_source', 'secondary_filter_cutoff_hz',
    ]
    display(batch_df[[c for c in _display_cols if c in batch_df.columns]].head(20))
else:
    print('⚠  No rows produced — check errors above.')

## Section 7 — Manual Validation Sidecar

Log manual peak corrections for scientific transparency.  
Each save appends one JSON entry to `derivatives/step_07_behavioral/manual_validation_log.json`.

> **When to use:** after running Section 5 (Visual Audit), note any peaks the algorithm  
> clearly missed (false negatives) or spuriously detected (false positives) — record the  
> time in seconds from the full recording timeline. This log is referenced at thesis defense  
> as evidence of human-in-the-loop QA on top of the algorithmic pipeline.

In [ ]:
# ── Cell 7: Section 7 — Manual Validation Sidecar ────────────────────────
import json as _json
from datetime import datetime, timezone

_LOG_PATH = STEP07_DIR / 'manual_validation_log.json'
# BATCH_SECONDARY_FILTER_HZ is defined in Cell 6; fall back gracefully if
# Cell 7 is run before Cell 6 in a partial session.
_batch_hz = globals().get('BATCH_SECONDARY_FILTER_HZ', float('nan'))

print('━' * 60)
print('SECTION 7  —  MANUAL VALIDATION SIDECAR')
print('━' * 60)
print('Record any peaks the algorithm clearly missed or falsely detected.')
print(f'Log file : {_LOG_PATH}')
print()

# ── Widgets ────────────────────────────────────────────────────────────────
_W_WIDE = widgets.Layout(width='620px')
_W_MED  = widgets.Layout(width='300px')

# Run ID pre-filled from the Section 2 dropdown
w_val_run = widgets.Text(
    value=w_recording.value or '',
    description='Run ID:',
    placeholder='Paste exact run_id from Section 2 dropdown',
    layout=_W_WIDE,
    style={'description_width': '130px'},
)

# Segment dropdown — prevents typos in a primary-key field
w_val_seg = widgets.Dropdown(
    options=['LeftHand', 'RightHand', 'LeftFoot', 'RightFoot', 'Head'],
    value='RightHand',
    description='Segment:',
    layout=_W_MED,
    style={'description_width': '130px'},
)

w_val_fp = widgets.Textarea(
    value='',
    description='False Positives:',
    placeholder='Comma-separated times in seconds — peaks algorithm detected that are NOT real  e.g. 12.3, 45.1',
    layout=widgets.Layout(width='620px', height='62px'),
    style={'description_width': '130px'},
)

w_val_fn = widgets.Textarea(
    value='',
    description='False Negatives:',
    placeholder='Comma-separated times in seconds — real peaks the algorithm MISSED  e.g. 22.0, 67.4',
    layout=widgets.Layout(width='620px', height='62px'),
    style={'description_width': '130px'},
)

w_val_notes = widgets.Textarea(
    value='',
    description='Notes:',
    placeholder='Optional — reason for correction, visual observations, confidence level, etc.',
    layout=widgets.Layout(width='620px', height='55px'),
    style={'description_width': '130px'},
)

save_btn_val = widgets.Button(
    description='Save to Log',
    button_style='success',
    layout=widgets.Layout(width='150px'),
    tooltip=f'Appends one JSON entry to {_LOG_PATH.name}',
)

clear_btn_val = widgets.Button(
    description='Clear Fields',
    button_style='warning',
    layout=widgets.Layout(width='120px'),
    tooltip='Reset text inputs (does not affect the log file)',
)

show_log_btn = widgets.Button(
    description='Show Log',
    button_style='',
    layout=widgets.Layout(width='120px'),
    tooltip='Display the full log as a DataFrame',
)

val_status = widgets.Output()


def _parse_times(text):
    out = []
    for tok in text.split(','):
        tok = tok.strip()
        if tok:
            try:
                out.append(float(tok))
            except ValueError:
                pass
    return sorted(out)


def _load_log():
    if _LOG_PATH.exists():
        try:
            with open(_LOG_PATH) as _f:
                data = _json.load(_f)
            return data if isinstance(data, list) else []
        except (_json.JSONDecodeError, OSError):
            return []
    return []


def _on_save_val(_b):
    with val_status:
        clear_output(wait=True)
        _rid  = w_val_run.value.strip()
        _seg  = w_val_seg.value
        _fp   = _parse_times(w_val_fp.value)
        _fn   = _parse_times(w_val_fn.value)
        _note = w_val_notes.value.strip()

        if not _rid:
            print('Run ID is empty — entry not saved.')
            return
        if not _fp and not _fn:
            print('No corrections entered (both FP and FN are empty) — entry not saved.')
            return

        _entry = {
            'timestamp_utc':             datetime.now(timezone.utc).isoformat(),
            'run_id':                    _rid,
            'segment':                   _seg,
            'false_positive_times_s':    _fp,
            'false_negative_times_s':    _fn,
            'notes':                     _note,
            'batch_secondary_filter_hz': _batch_hz,
            'pipeline_version':          'pipeline_V6.2',
        }

        _log = _load_log()
        _log.append(_entry)

        try:
            with open(_LOG_PATH, 'w') as _f:
                _json.dump(_log, _f, indent=2)
            print(f'Entry saved  ({len(_log)} total in {_LOG_PATH.name})')
            print(f'  Run ID         : {_rid}')
            print(f'  Segment        : {_seg}')
            print(f'  False positives: {_fp}  ({len(_fp)} rejected)')
            print(f'  False negatives: {_fn}  ({len(_fn)} added)')
            if _note:
                print(f'  Notes          : {_note}')
        except OSError as _exc:
            print(f'ERROR writing log: {_exc}')


def _on_clear_val(_b):
    w_val_fp.value    = ''
    w_val_fn.value    = ''
    w_val_notes.value = ''
    with val_status:
        clear_output(wait=True)
        print('Fields cleared.')


def _on_show_log(_b):
    with val_status:
        clear_output(wait=True)
        _log = _load_log()
        if not _log:
            print(f'Log is empty or does not exist yet: {_LOG_PATH}')
            return
        print(f'{len(_log)} entries in {_LOG_PATH.name}\n')
        _rows = [{
            'timestamp_utc': e.get('timestamp_utc', ''),
            'run_id':        e.get('run_id', ''),
            'segment':       e.get('segment', ''),
            'n_fp':          len(e.get('false_positive_times_s', [])),
            'n_fn':          len(e.get('false_negative_times_s', [])),
            'notes':         e.get('notes', ''),
        } for e in _log]
        display(pd.DataFrame(_rows))


save_btn_val.on_click(_on_save_val)
clear_btn_val.on_click(_on_clear_val)
show_log_btn.on_click(_on_show_log)

display(widgets.VBox([
    w_val_run,
    w_val_seg,
    w_val_fp,
    w_val_fn,
    w_val_notes,
    widgets.HBox([save_btn_val, clear_btn_val, show_log_btn]),
    val_status,
]))